# 🎵 EmojiBeats — Live Face Expression → Song Generator
### Jupyter Demo (No trained model needed — uses face-api.js via browser)

**What this does:**
1. Captures your webcam live in Jupyter
2. Detects your face expression using DeepFace (or mock for demo)
3. Maps expression → emoji
4. Searches a song via RapidAPI based on your emotion + language
5. Shows the song result + opens it

---
### 📦 Install dependencies first (run Cell 1)

In [1]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
import sys
!{sys.executable} -m pip install deepface opencv-python-headless requests ipywidgets Pillow tf_keras -q
print('✅ All packages installed!')

ERROR: virtualenv 20.34.0 has requirement typing-extensions>=4.13.2; python_version < "3.11", but you'll have typing-extensions 4.5.0 which is incompatible.
ERROR: tensorflow 2.13.1 has requirement keras<2.14,>=2.13.1, but you'll have keras 2.10.0 which is incompatible.
ERROR: tensorflow-cpu 2.10.0 has requirement protobuf<3.20,>=3.9.2, but you'll have protobuf 4.25.9 which is incompatible.
ERROR: tensorflow-cpu 2.10.0 has requirement tensorboard<2.11,>=2.10, but you'll have tensorboard 2.13.0 which is incompatible.
ERROR: tensorflow-cpu 2.10.0 has requirement tensorflow-estimator<2.11,>=2.10.0, but you'll have tensorflow-estimator 2.13.0 which is incompatible.
ERROR: selenium 4.27.1 has requirement typing_extensions~=4.9, but you'll have typing-extensions 4.5.0 which is incompatible.
ERROR: pydantic 2.10.6 has requirement typing-extensions>=4.12.2, but you'll have typing-extensions 4.5.0 which is incompatible.
ERROR: pydantic-core 2.27.2 has requirement typing-extensions!=4.7.0,>=4.6.

In [ ]:
# ── CELL 2: Imports & Config ──────────────────────────────────────────────────
import cv2
import numpy as np
import requests
import base64
import time
import json
import threading
import collections
import webbrowser
from IPython.display import display, HTML, clear_output, Image as IPImage
from PIL import Image
import io
import ipywidgets as widgets

# ╔══════════════════════════════════════════════╗
# ║        🔑  PASTE YOUR RAPIDAPI KEY HERE      ║
# ╚══════════════════════════════════════════════╝
RAPIDAPI_KEY = "ad7aedaa34msh57cb97ff1ebaf7ap184a0djsn5c0a4ca44f30"

# API choice: 'spotify23' | 'deezer' | 'shazam' | 'ytmusic'
API_CHOICE   = 'spotify23'

# ── EMOTION → EMOJI MAP ───────────────────────────────────────────────────────
EMOTION_EMOJI = {
    'happy':    ('😄', 'Happy',    (0, 210, 90)),
    'sad':      ('😢', 'Sad',      (255, 80, 60)),
    'angry':    ('😠', 'Angry',    (30, 40, 255)),
    'fear':     ('😨', 'Fearful',  (255, 200, 0)),
    'surprise': ('😲', 'Surprised',(0, 200, 255)),
    'disgust':  ('🤢', 'Disgusted',(200, 0, 200)),
    'neutral':  ('😐', 'Neutral',  (180, 180, 180)),
}

# ── LANGUAGE → SEARCH QUERIES ─────────────────────────────────────────────────
EMOTION_SEARCH = {
    'English': {
        'happy': 'upbeat happy pop feel good',
        'sad': 'sad emotional ballad heartbreak',
        'angry': 'rock rage intense energy',
        'fear': 'calm peaceful ambient',
        'surprise': 'exciting energetic pop dance',
        'disgust': 'dark alternative indie',
        'neutral': 'lofi chill study beats',
    },
    'Tamil': {
        'happy': 'tamil kuthu happy dance',
        'sad': 'tamil sad melody love',
        'angry': 'tamil mass bgm action',
        'fear': 'tamil devotional calm',
        'surprise': 'tamil party dance number',
        'disgust': 'tamil independent music',
        'neutral': 'tamil soft melody evening',
    },
    'Hindi': {
        'happy': 'bollywood happy dance party',
        'sad': 'hindi sad romantic melody',
        'angry': 'hindi intense bgm',
        'fear': 'hindi devotional spiritual',
        'surprise': 'bollywood party 2024',
        'disgust': 'hindi indie alternative',
        'neutral': 'hindi soft ghazal',
    },
    'Telugu': {
        'happy': 'telugu happy dance songs',
        'sad': 'telugu sad melody love',
        'angry': 'telugu mass bgm action',
        'fear': 'telugu devotional calm',
        'surprise': 'telugu energetic dance',
        'disgust': 'telugu indie folk',
        'neutral': 'telugu soft melody',
    },
    'Korean': {
        'happy': 'kpop upbeat happy dance',
        'sad': 'kpop ballad sad emotional',
        'angry': 'k-rock intense energy',
        'fear': 'korean healing soft',
        'surprise': 'kpop exciting new',
        'disgust': 'korean indie alternative',
        'neutral': 'korean lofi chill',
    },
    'Spanish': {
        'happy': 'reggaeton feliz fiesta',
        'sad': 'balada romantica triste',
        'angry': 'rock español intenso',
        'fear': 'musica relajante calma',
        'surprise': 'salsa bachata energetic',
        'disgust': 'indie alternativo oscuro',
        'neutral': 'bossa nova suave',
    },
}

print('✅ Config loaded!')
print(f'   API: {API_CHOICE}  |  Languages: {list(EMOTION_SEARCH.keys())}')

In [ ]:
# ── CELL 3: RapidAPI Search Function ─────────────────────────────────────────

API_CONFIGS = {
    'spotify23': {
        'url':    'https://spotify23.p.rapidapi.com/search/',
        'host':   'spotify23.p.rapidapi.com',
        'params': lambda q: {'q': q, 'type': 'tracks', 'offset': '0', 'limit': '5'},
        'parse':  lambda d: [
            {
                'title':  t['data']['name'],
                'artist': t['data']['artists']['items'][0]['profile']['name'],
                'url':    'https://open.spotify.com/track/' + t['data']['id'],
                'cover':  (t['data'].get('albumOfTrack',{}).get('coverArt',{}).get('sources',[{}]) or [{}])[0].get('url',''),
            }
            for t in d.get('tracks',{}).get('items',[]) if t.get('data')
        ],
    },
    'deezer': {
        'url':    'https://deezerdevs-deezer.p.rapidapi.com/search',
        'host':   'deezerdevs-deezer.p.rapidapi.com',
        'params': lambda q: {'q': q},
        'parse':  lambda d: [
            {
                'title':   t['title'],
                'artist':  t['artist']['name'],
                'url':     t['link'],
                'cover':   t.get('album',{}).get('cover_medium',''),
                'preview': t.get('preview',''),
            }
            for t in d.get('data',[])
        ],
    },
    'shazam': {
        'url':    'https://shazam-core.p.rapidapi.com/v1/search/multi',
        'host':   'shazam-core.p.rapidapi.com',
        'params': lambda q: {'search_type': 'SONGS', 'query': q},
        'parse':  lambda d: [
            {
                'title':  t['heading']['title'],
                'artist': t['heading']['subtitle'],
                'url':    t.get('stores',{}).get('apple',{}).get('trackurl','#'),
                'cover':  t.get('images',{}).get('coverart',''),
            }
            for t in d.get('tracks',{}).get('hits',[]) if t.get('heading')
        ],
    },
    'ytmusic': {
        'url':    'https://ytmusic-music-player.p.rapidapi.com/search',
        'host':   'ytmusic-music-player.p.rapidapi.com',
        'params': lambda q: {'query': q},
        'parse':  lambda d: [
            {
                'title':  t.get('title','Unknown'),
                'artist': t.get('artist',{}).get('name','Unknown') if isinstance(t.get('artist'),dict) else str(t.get('artist','?')),
                'url':    f"https://www.youtube.com/watch?v={t.get('videoId','')}",
                'cover':  '',
            }
            for t in (d if isinstance(d,list) else [])
        ],
    },
}

def search_tracks(query, api=API_CHOICE, key=RAPIDAPI_KEY):
    """Call RapidAPI and return list of track dicts."""
    if key == 'YOUR_RAPIDAPI_KEY_HERE':
        print('⚠  No RapidAPI key! Returning mock results for demo.')
        return [
            {'title': f'Demo Track {i+1} ({query[:20]})', 'artist': 'Demo Artist',
             'url': 'https://open.spotify.com', 'cover': ''}
            for i in range(3)
        ]
    cfg = API_CONFIGS[api]
    try:
        r = requests.get(
            cfg['url'],
            headers={'X-RapidAPI-Key': key, 'X-RapidAPI-Host': cfg['host']},
            params=cfg['params'](query),
            timeout=8
        )
        r.raise_for_status()
        return cfg['parse'](r.json())[:5]
    except Exception as e:
        print(f'❌ RapidAPI error: {e}')
        return []

print('✅ RapidAPI functions ready!')

In [4]:
# ── CELL 4: Face Expression Detection (DeepFace) ──────────────────────────────

try:
    from deepface import DeepFace
    DEEPFACE_OK = True
    print('✅ DeepFace loaded — REAL emotion detection active!')
except ImportError:
    DEEPFACE_OK = False
    print('⚠  DeepFace not available — using MOCK detection for demo')
    print('   Install: pip install deepface tf_keras')


def detect_emotion_from_frame(frame_bgr):
    """
    Returns (emotion_str, confidence_float, all_scores_dict)
    Uses DeepFace if available, otherwise mock random for demo.
    """
    if DEEPFACE_OK:
        try:
            result = DeepFace.analyze(
                frame_bgr,
                actions=['emotion'],
                enforce_detection=False,  # don't crash if no face
                silent=True
            )
            if isinstance(result, list):
                result = result[0]
            emotion   = result['dominant_emotion'].lower()
            scores    = {k.lower(): v/100 for k, v in result['emotion'].items()}
            # Map DeepFace labels to our labels
            label_map = {'angry':'angry','disgust':'disgust','fear':'fear',
                         'happy':'happy','sad':'sad','surprise':'surprise','neutral':'neutral'}
            emotion = label_map.get(emotion, 'neutral')
            conf    = scores.get(emotion, 0.0)
            return emotion, conf, scores
        except Exception as e:
            pass  # fall through to mock

    # ── MOCK for demo when DeepFace unavailable ──────────────────
    emotions = ['happy','sad','angry','neutral','surprise','fear','disgust']
    weights  = [0.35, 0.20, 0.10, 0.20, 0.08, 0.04, 0.03]
    emotion  = np.random.choice(emotions, p=weights)
    conf     = np.random.uniform(0.55, 0.95)
    scores   = {e: np.random.uniform(0, 0.1) for e in emotions}
    scores[emotion] = conf
    return emotion, conf, scores


print('✅ Emotion detector ready!')

2026-03-30 15:26:47.542587: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


TypeError: Descriptors cannot be created directly.
If this call came from a _pb2.py file, your generated code is out of date and must be regenerated with protoc >= 3.19.0.
If you cannot immediately regenerate your protos, some other possible workarounds are:
 1. Downgrade the protobuf package to 3.20.x or lower.
 2. Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python (but this will use pure-Python parsing and will be much slower).

More information: https://developers.google.com/protocol-buffers/docs/news/2022-05-06#python-updates

In [ ]:
# ── CELL 5: HTML Display Helpers ──────────────────────────────────────────────

def frame_to_base64(frame_bgr, quality=85):
    """Convert OpenCV frame to base64 JPEG string for HTML display."""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    buf = io.BytesIO()
    pil.save(buf, format='JPEG', quality=quality)
    return base64.b64encode(buf.getvalue()).decode('utf-8')


def draw_emotion_overlay(frame, emotion, conf, scores, face_box=None):
    """Draw emotion info on frame."""
    h, w = frame.shape[:2]
    out  = frame.copy()

    if emotion and emotion in EMOTION_EMOJI:
        emoji_char, label, (r,g,b) = EMOTION_EMOJI[emotion]
        color = (b, g, r)  # OpenCV BGR

        # Face bounding box (if detected)
        if face_box:
            x,y,fw,fh = face_box
            corner = 18
            for (px,py),(qx,qy) in [
                ((x,y),(x+corner,y)),((x,y),(x,y+corner)),
                ((x+fw,y),(x+fw-corner,y)),((x+fw,y),(x+fw,y+corner)),
                ((x,y+fh),(x+corner,y+fh)),((x,y+fh),(x,y+fh-corner)),
                ((x+fw,y+fh),(x+fw-corner,y+fh)),((x+fw,y+fh),(x+fw,y+fh-corner)),
            ]:
                cv2.line(out, (px,py), (qx,qy), color, 2)

        # Top bar
        overlay = out.copy()
        cv2.rectangle(overlay, (0,0), (w,60), (15,15,25), -1)
        cv2.addWeighted(overlay, 0.75, out, 0.25, 0, out)

        cv2.putText(out, 'EMOJIBEATS LIVE', (12,22),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (80,200,100), 1, cv2.LINE_AA)

        emo_text = f'{label.upper()}  {conf*100:.0f}%'
        cv2.putText(out, emo_text, (12,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)

        # Probability mini bars (right side)
        bar_x = w - 135
        for i, (emo, prob) in enumerate(sorted(scores.items(), key=lambda x:-x[1])):
            by  = 8 + i * 18
            bw  = int(prob * 110)
            c   = EMOTION_EMOJI.get(emo, ('','',( 150,150,150)))[2]
            col = (c[2], c[1], c[0])  # RGB→BGR
            cv2.rectangle(out, (bar_x, by), (bar_x+bw, by+12), col, -1)
            cv2.putText(out, f'{emo[:3]} {prob*100:.0f}%',
                        (bar_x+115, by+10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.3, col, 1)

        # Confidence bar bottom
        overlay2 = out.copy()
        cv2.rectangle(overlay2, (0,h-30),(w,h),(15,15,25),-1)
        cv2.addWeighted(overlay2, 0.7, out, 0.3, 0, out)
        bar_w = int((w-20)*conf)
        cv2.rectangle(out, (10,h-20),(w-10,h-8),(40,40,55),-1)
        cv2.rectangle(out, (10,h-20),(10+bar_w,h-8), color,-1)

    return out


def render_song_card(tracks, emotion, language):
    """Render HTML song result cards."""
    if not tracks:
        return '<p style="color:#f472b6">⚠ No tracks found</p>'

    emoji, label, (r,g,b) = EMOTION_EMOJI.get(emotion, ('😐','Neutral',(180,180,180)))
    color = f'rgb({r},{g},{b})'

    cards = ''.join([
        f'''
        <div style="
            background:#16161f;
            border:1px solid #2a2a40;
            border-radius:12px;
            padding:14px 18px;
            margin-bottom:10px;
            display:flex;
            align-items:center;
            gap:14px;
            font-family: monospace;
        ">
            {'<img src="'+t["cover"]+'" style="width:52px;height:52px;border-radius:8px;object-fit:cover" />' if t.get('cover') else '<div style="width:52px;height:52px;background:#252535;border-radius:8px;display:flex;align-items:center;justify-content:center;font-size:22px">🎵</div>'}
            <div style="flex:1">
                <div style="color:#eeeef8;font-size:14px;font-weight:bold">{t["title"]}</div>
                <div style="color:#7070a0;font-size:12px;margin-top:3px">{t["artist"]}</div>
            </div>
            <a href="{t['url']}" target="_blank" style="
                background:{color};
                color:#000;
                padding:8px 16px;
                border-radius:100px;
                font-size:12px;
                font-weight:bold;
                text-decoration:none;
                letter-spacing:1px;
            ">▶ PLAY</a>
        </div>
        '''
        for t in tracks
    ])

    return f'''
    <div style="background:#0a0a0f;padding:20px;border-radius:16px;border:1px solid #252535">
        <div style="
            font-family:monospace;
            font-size:11px;
            letter-spacing:3px;
            color:{color};
            text-transform:uppercase;
            margin-bottom:16px;
        ">{emoji} {label} VIBES · {language.upper()} SONGS</div>
        {cards}
    </div>
    '''

print('✅ Display helpers ready!')

In [ ]:
# ── CELL 6: 🎬 LIVE DEMO — Run this cell! ────────────────────────────────────
# Captures N frames from webcam, detects emotion, searches song via RapidAPI
# Press the STOP button (■) in Jupyter to end early

# ── Settings ─────────────────────────────────────────────────────────────────
LANGUAGE      = 'Tamil'     # ← change: English / Tamil / Hindi / Telugu / Korean / Spanish
CAPTURE_SECS  = 6           # how many seconds to scan your face
DISPLAY_FPS   = 4           # frames per second shown in notebook (keep low)
SMOOTH_WINDOW = 8           # rolling vote window for stable emotion

# ── Validate ──────────────────────────────────────────────────────────────────
if LANGUAGE not in EMOTION_SEARCH:
    print(f'⚠ Language "{LANGUAGE}" not found. Defaulting to English.')
    LANGUAGE = 'English'

# ── UI Widgets ────────────────────────────────────────────────────────────────
frame_out    = widgets.Output()
status_out   = widgets.Output()
result_out   = widgets.Output()

progress_bar = widgets.IntProgress(
    value=0, min=0, max=CAPTURE_SECS,
    description='Scanning:',
    bar_style='success',
    style={'bar_color': '#1db954'},
    layout=widgets.Layout(width='100%')
)

lang_label = widgets.HTML(
    value=f'<span style="font-family:monospace;color:#7070a0;font-size:12px;letter-spacing:2px">'
          f'LANGUAGE: <span style="color:#1db954">{LANGUAGE.upper()}</span> · '
          f'API: <span style="color:#c084fc">{API_CHOICE.upper()}</span></span>'
)

display(lang_label, progress_bar, frame_out, status_out, result_out)

# ── Run capture ───────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    print('❌ Cannot open webcam. Check camera permissions.')
else:
    emotion_history = collections.deque(maxlen=SMOOTH_WINDOW)
    conf_history    = collections.deque(maxlen=SMOOTH_WINDOW)
    frame_delay     = 1.0 / DISPLAY_FPS
    t_start         = time.time()
    last_display    = 0
    final_emotion   = 'neutral'
    final_conf      = 0.0
    frame_count     = 0

    try:
        while True:
            elapsed = time.time() - t_start
            if elapsed >= CAPTURE_SECS:
                break

            ret, frame = cap.read()
            if not ret:
                time.sleep(0.05)
                continue

            frame = cv2.flip(frame, 1)  # mirror
            frame_count += 1

            # Detect emotion every frame
            emotion, conf, scores = detect_emotion_from_frame(frame)
            emotion_history.append(emotion)
            conf_history.append(conf)

            # Stable emotion = majority vote
            stable = collections.Counter(emotion_history).most_common(1)[0][0]
            avg_conf = float(np.mean(conf_history))
            final_emotion = stable
            final_conf    = avg_conf

            # Update progress bar
            progress_bar.value = int(elapsed)

            # Display frame at capped FPS
            now = time.time()
            if now - last_display >= frame_delay:
                last_display = now
                annotated = draw_emotion_overlay(frame, stable, avg_conf, scores)
                b64 = frame_to_base64(annotated, quality=80)

                with frame_out:
                    clear_output(wait=True)
                    display(HTML(
                        f'<img src="data:image/jpeg;base64,{b64}" '
                        f'style="width:100%;max-width:560px;border-radius:12px;'
                        f'border:2px solid #252535" />'
                    ))

                with status_out:
                    clear_output(wait=True)
                    emoji, label, _ = EMOTION_EMOJI.get(stable, ('😐','Neutral',(180,180,180)))
                    display(HTML(
                        f'<div style="font-family:monospace;color:#5a5a80;font-size:11px;'
                        f'letter-spacing:2px;margin:6px 0">'
                        f'FRAME {frame_count} · {elapsed:.1f}s / {CAPTURE_SECS}s · '
                        f'DETECTED: <span style="color:#1db954">{emoji} {label.upper()} ({avg_conf*100:.0f}%)</span>'
                        f'</div>'
                    ))

    except KeyboardInterrupt:
        print('⏹ Stopped early.')
    finally:
        cap.release()

    progress_bar.value = CAPTURE_SECS
    progress_bar.bar_style = 'info'

    # ── Final emotion decided ─────────────────────────────────────────────────
    emoji, label, (r,g,b) = EMOTION_EMOJI.get(final_emotion, ('😐','Neutral',(180,180,180)))

    with status_out:
        clear_output(wait=True)
        display(HTML(f'''
        <div style="
            font-family:monospace;
            background:#0a0a0f;
            border:1px solid rgb({r},{g},{b});
            border-radius:12px;
            padding:16px 20px;
            margin:10px 0;
            box-shadow: 0 0 20px rgba({r},{g},{b},0.2);
        ">
            <div style="font-size:36px;margin-bottom:6px">{emoji}</div>
            <div style="color:rgb({r},{g},{b});font-size:18px;letter-spacing:3px">{label.upper()}</div>
            <div style="color:#5a5a80;font-size:11px;margin-top:4px;letter-spacing:2px">
                CONFIDENCE: {final_conf*100:.0f}% · FRAMES ANALYZED: {frame_count}
            </div>
        </div>
        '''))

    # ── Search songs ─────────────────────────────────────────────────────────
    query = EMOTION_SEARCH[LANGUAGE].get(final_emotion, f'{final_emotion} music')
    print(f'🔍 Searching [{API_CHOICE}]: "{query}" ...')

    tracks = search_tracks(query)

    with result_out:
        clear_output(wait=True)
        if tracks:
            display(HTML(render_song_card(tracks, final_emotion, LANGUAGE)))
        else:
            display(HTML('<p style="color:#f472b6;font-family:monospace">⚠ No tracks found — check your RapidAPI key</p>'))

    print(f'\n✅ Done! Emotion: {label} | Language: {LANGUAGE} | {len(tracks)} tracks found')

In [ ]:
# ── CELL 7: Quick Test — Search without webcam ────────────────────────────────
# Useful to test your RapidAPI key is working correctly

TEST_EMOTION  = 'happy'
TEST_LANGUAGE = 'Tamil'

query  = EMOTION_SEARCH[TEST_LANGUAGE][TEST_EMOTION]
print(f'🔍 Test search: "{query}"')

tracks = search_tracks(query)

if tracks:
    print(f'\n✅ Found {len(tracks)} tracks:\n')
    for i, t in enumerate(tracks):
        print(f'  {i+1}. {t["title"]} — {t["artist"]}')
        print(f'     🔗 {t["url"]}')
else:
    print('\n⚠ No tracks returned. Check your RAPIDAPI_KEY in Cell 2.')

display(HTML(render_song_card(tracks, TEST_EMOTION, TEST_LANGUAGE)))

In [ ]:
# ── CELL 8: Snapshot — Capture single frame & detect ─────────────────────────
# Useful if live loop is too slow on your machine

LANGUAGE_SNAP = 'Hindi'   # ← change language here

cap = cv2.VideoCapture(0)
time.sleep(0.5)            # warm up camera
ret, frame = cap.read()
cap.release()

if not ret:
    print('❌ Could not capture frame.')
else:
    frame = cv2.flip(frame, 1)
    emotion, conf, scores = detect_emotion_from_frame(frame)
    annotated = draw_emotion_overlay(frame, emotion, conf, scores)

    emoji, label, (r,g,b) = EMOTION_EMOJI.get(emotion, ('😐','Neutral',(180,180,180)))
    b64 = frame_to_base64(annotated)

    display(HTML(f'''
    <div style="font-family:monospace;max-width:580px">
        <img src="data:image/jpeg;base64,{b64}"
             style="width:100%;border-radius:12px;border:2px solid rgb({r},{g},{b})" />
        <div style="margin-top:10px;color:rgb({r},{g},{b});font-size:16px;letter-spacing:3px">
            {emoji} {label.upper()} — {conf*100:.0f}% confidence
        </div>
    </div>
    '''))

    # Search song for snapshot emotion
    query  = EMOTION_SEARCH.get(LANGUAGE_SNAP, EMOTION_SEARCH['English']).get(emotion, emotion)
    print(f'\n🔍 Searching: "{query}"')
    tracks = search_tracks(query)
    display(HTML(render_song_card(tracks, emotion, LANGUAGE_SNAP)))